In [1]:
# ============================================================
# [CELL 00] — CONFIG & IMPORTS
# Projet POKEMON — Référentiel Universel FR
# Prismora Solutions — v2.0 — 2026
# Notebook : 02_POCKET_CARDS.ipynb
# ============================================================

import requests
import pandas as pd
import json
import re
import time
from pathlib import Path
from bs4 import BeautifulSoup
from IPython.display import display

# Répertoire racine = dossier du notebook
ROOT = Path.cwd()
CONFIG_PATH = ROOT / "config" / "sources.json"

# Chargement config
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

SOURCE_PRIMARY   = CONFIG["sources"]["primary"]
SOURCE_SECONDARY = CONFIG["sources"]["secondary"]
SOURCE_TERTIARY  = CONFIG["sources"]["tertiary"]

# Chargement sets_master (construit par 01_SETUP_BDD)
SETS_MASTER_PATH = ROOT / CONFIG["output"]["sets_master"]
df_sets = pd.read_csv(SETS_MASTER_PATH, encoding="utf-8-sig")

# Contrôle
print("=" * 55)
print("  PROJET POKEMON — Cartes TCG Pocket FR")
print("  Prismora Solutions — 02_POCKET_CARDS.ipynb")
print("=" * 55)
print(f"\n📁 Racine       : {ROOT}")
print(f"✅ Config        : {SOURCE_PRIMARY['name']}")
print(f"✅ sets_master   : {len(df_sets)} sets chargés")
print(f"\n📋 Sets à traiter :")
display(df_sets[["code_set", "nom_fr", "nb_cartes"]].to_string(index=False))
print(f"\n📊 Total cartes attendues : {df_sets['nb_cartes'].sum()}")

  PROJET POKEMON — Cartes TCG Pocket FR
  Prismora Solutions — 02_POCKET_CARDS.ipynb

📁 Racine       : i:\Drive partagés\Prismora - Drive Principal\01_Clients\POKEMON
✅ Config        : TCGdex API (FR natif)
✅ sets_master   : 19 sets chargés

📋 Sets à traiter :


"code_set                    nom_fr  nb_cartes\n      A1       Puissance Génétique        286\n PROMO-A                   Promo-A        117\n     A1a           L’Île Fabuleuse         86\n      A2      Choc Spatio-Temporel        207\n     A2a        Lumière Triomphale         96\n     A2b Réjouissances Rayonnantes        111\n      A3          Gardiens Astraux        239\n     A3a Crise Interdimensionnelle        103\n     A3b      La Clairière d'Évoli        107\n      A4 Sagesse Entre Ciel et Mer        241\n     A4a            Source Secrète        105\n     A4b        Booster de Luxe ex        379\n      B1            Méga-Ascension        331\n PROMO-B                   Promo-B         62\n     B1a      Embrasement Écarlate        103\n      B2           Parade Onirique        234\n     B2a      Merveilles de Paldea        131\n     B2b          Méga-Rayonnement        117\n      B3           Aura Palpitante        234"


📊 Total cartes attendues : 3289


In [2]:
# ============================================================
# [CELL 01] — RÉCUPÉRATION CARTES POCKET (TCGdex FR + flibustier)
# TCGdex FR en primary | flibustier en fallback si 0 cartes ou 404
# PROMO-A et PROMO-B forcés sur flibustier (TCGdex incomplet)
# ============================================================

# Sets à forcer sur flibustier
FORCER_FLIB = ["PROMO-A", "PROMO-B"]

# Tous les sets (y compris PROMO-B à 0 dans sets_master)
sets_actifs = df_sets.copy()
all_cards_index = {}
source_par_set = {}

for _, row in sets_actifs.iterrows():
    code_set    = row["code_set"]
    code_tcgdex = row["code_tcgdex"]
    nom_fr      = row["nom_fr"]

    cards = []

    # --- Forçage flibustier ou tentative TCGdex FR ---
    if code_set not in FORCER_FLIB:
        try:
            url = f"{SOURCE_PRIMARY['base_url']}/sets/{code_tcgdex}"
            resp = requests.get(url, timeout=15)
            resp.raise_for_status()
            cards = resp.json().get("cards", [])
        except:
            pass

    # --- Fallback flibustier si vide, erreur ou forcé ---
    if len(cards) == 0:
        try:
            url_flib = f"{SOURCE_SECONDARY['base_url']}/cards/{code_set}.json"
            resp = requests.get(url_flib, timeout=15)
            resp.raise_for_status()
            cards = resp.json()
            source_par_set[code_set] = "flibustier"
        except Exception as e:
            source_par_set[code_set] = f"ERREUR"
            print(f"  ❌ {code_set:10s} | {nom_fr:30s} | ERREUR : {e}")
            continue
    else:
        source_par_set[code_set] = "tcgdex"

    all_cards_index[code_set] = cards
    print(f"  ✅ {code_set:10s} | {nom_fr:30s} | {len(cards):4d} cartes | {source_par_set[code_set]}")
    time.sleep(0.2)

# Sauvegarde brute
with open(ROOT / "data/raw/cards_index_raw.json", "w", encoding="utf-8") as f:
    json.dump(all_cards_index, f, ensure_ascii=False, indent=2)

# Contrôle
total_recupere  = sum(len(v) for v in all_cards_index.values())
total_attendu   = df_sets["nb_cartes"].sum()
nb_tcgdex       = sum(1 for s in source_par_set.values() if s == "tcgdex")
nb_flib         = sum(1 for s in source_par_set.values() if s == "flibustier")
nb_erreurs      = sum(1 for s in source_par_set.values() if s == "ERREUR")
ecart           = total_recupere - total_attendu

print(f"\n{'='*45}")
print(f"  Sets traités   : {len(all_cards_index)}/19")
print(f"  TCGdex FR      : {nb_tcgdex} sets")
print(f"  flibustier     : {nb_flib} sets")
print(f"  Erreurs        : {nb_erreurs} sets")
print(f"{'='*45}")
print(f"  Cartes récupérées : {total_recupere}")
print(f"  Cartes attendues  : {total_attendu} (sets_master)")
print(f"  Écart             : {ecart:+d}")
if abs(ecart) < 50:
    print(f"  ✅ Écart acceptable")
else:
    print(f"  ⚠️  Écart important — vérifier les sets concernés")
print(f"{'='*45}")

  ✅ A1         | Puissance Génétique            |  286 cartes | tcgdex
  ✅ PROMO-A    | Promo-A                        |  117 cartes | flibustier
  ✅ A1a        | L’Île Fabuleuse                |   86 cartes | tcgdex
  ✅ A2         | Choc Spatio-Temporel           |  207 cartes | tcgdex
  ✅ A2a        | Lumière Triomphale             |   96 cartes | tcgdex
  ✅ A2b        | Réjouissances Rayonnantes      |  111 cartes | tcgdex
  ✅ A3         | Gardiens Astraux               |  239 cartes | tcgdex
  ✅ A3a        | Crise Interdimensionnelle      |  103 cartes | tcgdex
  ✅ A3b        | La Clairière d'Évoli           |  107 cartes | tcgdex
  ✅ A4         | Sagesse Entre Ciel et Mer      |  241 cartes | tcgdex
  ✅ A4a        | Source Secrète                 |  105 cartes | tcgdex
  ✅ A4b        | Booster de Luxe ex             |  379 cartes | flibustier
  ✅ B1         | Méga-Ascension                 |  331 cartes | flibustier
  ✅ PROMO-B    | Promo-B                        |   62 cartes | f